In [1]:
import pandas as pd

score_sample = pd.read_parquet("/Users/nadia/Desktop/redditRun_june/comment_data/score_sample.parquet")
COMMENTS_PER_CALL = 10

for chunk_idx in [5212, 5328]:
    start = chunk_idx * COMMENTS_PER_CALL
    end = start + COMMENTS_PER_CALL
    chunk = score_sample.iloc[start:end]
    print(f"\n=== chunk {chunk_idx} ===")
    for _, row in chunk.iterrows():
        print(f"[{row['id']}] len={len(str(row['body']))}: {str(row['body'])[:150]!r}")


=== chunk 5212 ===
[kswxrmx] len=411: '> I’ve been in IT for 20 years and recently switched from upper mgmt in a F100 enterprise space to SMB as I was burned out. \n\nMy 2 cents is that you c'
[o8adnn0] len=58: '1) disable IPv6\n2) unplug it from the network and ping it.'
[jd9egly] len=55: 'Put an Azure sticker on it, then they wil start to care'
[kcl5zpf] len=147: 'Let it burn. Only your account has admin access and here is the password…and due to hr deprovisioning policy, the account has now been disabled. /s'
[fc33l44] len=1067: "I've gone from the Ops side to DevOps and weekly more and more dev work these days.\n\nI studied via PluralSight, made internal applications and website"
[g4n3u61] len=265: 'Healthcare IT is a nightmare. I spent a year trying to make things run smoothly and without cryptovirusses. I plan, they deny. Then, when all data is '
[gj1f4wo] len=1419: 'I asked a similar question on the Spiceworks forum, and got a very good response I thought.  All credit to Ken525

In [2]:
import json
frozen = json.load(open("/Users/nadia/Desktop/redditRun_june/comment_data/frozen_concepts.json"))
for c in frozen:
    print(f"{c['name']}: {len(c['prompt'])} chars")

Workplace Problem Guidance: 172 chars
Career Planning Advice: 168 chars
Emotional Support: 251 chars
Critical Pushback: 163 chars
Personal Relating: 146 chars
Discussion Direction: 156 chars
Practical Advice: 166 chars
Support Connections: 173 chars


In [3]:
import pandas as pd
score_sample = pd.read_parquet("/Users/nadia/Desktop/redditRun_june/comment_data/score_sample.parquet")

for chunk_idx in [372, 542, 765, 1615]:
    start, end = chunk_idx * 10, chunk_idx * 10 + 10
    total_chars = score_sample.iloc[start:end]["body"].str.len().sum()
    print(f"chunk {chunk_idx}: {total_chars:,} total chars")

chunk 372: 3,070 total chars
chunk 542: 4,276 total chars
chunk 765: 8,055 total chars
chunk 1615: 4,450 total chars


In [4]:
import pandas as pd
score_sample = pd.read_parquet("/Users/nadia/Desktop/redditRun_june/comment_data/score_sample.parquet")
chunk_ids = score_sample.iloc[5212*10:5212*10+10]["id"].tolist()

# Paste in the actual scores dict from one of the "9/10 answered" lines above
answered = {'kswxrmx': 1, 'o8adnn0': 1, 'jd9egly': 0, 'kcl5zpf': 0, 'fc33l44': 1,
            'g4n3u61': 0, 'gj1f4wo': 1, 'nz6i5aq': 0, 'ihhh2ai': 1}  # example -- use your real one

missing = set(chunk_ids) - set(answered.keys())
print("Missing comment ID:", missing)
print(score_sample[score_sample["id"].isin(missing)]["body"].values)

Missing comment ID: {'hhz5yu3'}
['WTF is wrong with you using CCTV footage for an incident report because a user couldn´t print and lied to her manager?\n\nThis one little mail to her manager to explain what happened. Case closed.']


In [5]:
import pandas as pd

wide = pd.read_parquet("/Users/nadia/Desktop/redditRun_june/comment_data/score_sample_wide.parquet")

print(f"Shape: {wide.shape}")
print(f"Columns: {wide.columns.tolist()}\n")

sample_10 = wide.sample(10, random_state=1)
print(sample_10)

sample_10.to_csv("/Users/nadia/Desktop/redditRun_june/comment_data/score_sample_wide_preview.csv", index=False)
print("\nSaved: score_sample_wide_preview.csv")

Shape: (53283, 18)
Columns: ['id', 'Workplace Problem Guidance', 'Career Planning Advice', 'Emotional Support', 'Critical Pushback', 'Personal Relating', 'Discussion Direction', 'Practical Advice', 'Support Connections', 'informational_support', 'emotional_support', 'esteem_support', 'tangible_support', 'network_support', 'unsupportive_response', 'subreddit_source', 'post_id', 'w']

            id  Workplace Problem Guidance  Career Planning Advice  \
33163  lgc5exa                         1.0                     1.0   
14690  iipa2i7                         0.0                     0.0   
13381  iansn36                         1.0                     0.0   
24936  k0kciqi                         1.0                     1.0   
108    dtjcr0p                         0.0                     0.0   
38193  mc3tm6m                         0.0                     0.0   
35687  lvjqg1b                         1.0                     0.0   
4785   ffwo4rp                         0.0            

In [6]:
import json
import pandas as pd

path = "/Users/nadia/Desktop/redditRun_june/comment_data/score_results/scores.jsonl"

# Dedupe by (label_id, chunk_idx), same logic as inspect_scores.py, but
# filtered to just this one label
latest_by_chunk = {}
with open(path) as f:
    for line in f:
        row = json.loads(line)
        if row["label_name"] != "Discussion Direction":
            continue
        latest_by_chunk[row["chunk_idx"]] = row

total = len(latest_by_chunk)
full = sum(1 for r in latest_by_chunk.values() if len(r["scores"]) == 10)
partial = sum(1 for r in latest_by_chunk.values() if 0 < len(r["scores"]) < 10)
empty = sum(1 for r in latest_by_chunk.values() if len(r["scores"]) == 0)

print(f"Discussion Direction: {total:,} distinct chunks")
print(f"  Full: {full:,} ({full/total:.1%})")
print(f"  Partial: {partial:,} ({partial/total:.1%})")
print(f"  Empty: {empty:,} ({empty/total:.1%})")

print(f"\nMissing chunk_idx values (never scored at all): "
      f"{5329 - total} out of 5,329 expected")

# Show a few empty/partial ones
shown = 0
for chunk_idx, row in sorted(latest_by_chunk.items()):
    if len(row["scores"]) < 10 and shown < 10:
        print(f"  chunk {chunk_idx}: {len(row['scores'])}/10 -- {row['scores']}")
        shown += 1

Discussion Direction: 5,212 distinct chunks
  Full: 5,170 (99.2%)
  Partial: 2 (0.0%)
  Empty: 40 (0.8%)

Missing chunk_idx values (never scored at all): 117 out of 5,329 expected
  chunk 92: 0/10 -- {}
  chunk 112: 0/10 -- {}
  chunk 234: 0/10 -- {}
  chunk 772: 0/10 -- {}
  chunk 1551: 0/10 -- {}
  chunk 1668: 0/10 -- {}
  chunk 1689: 0/10 -- {}
  chunk 1775: 0/10 -- {}
  chunk 1840: 0/10 -- {}
  chunk 1880: 0/10 -- {}


In [7]:
import json
import pandas as pd

wide_path = "/Users/nadia/Desktop/redditRun_june/comment_data/score_sample_wide.parquet"
jsonl_path = "/Users/nadia/Desktop/redditRun_june/comment_data/score_results/scores.jsonl"

wide = pd.read_parquet(wide_path)
label_cols = [c for c in wide.columns if c not in ["id", "subreddit_source", "post_id", "w"]]

# How many distinct chunks exist per label in the raw file (dedup by label+chunk)
latest = {}
with open(jsonl_path) as f:
    for line in f:
        row = json.loads(line)
        latest[(row["label_name"], row["chunk_idx"])] = row

chunks_seen_per_label = {}
for (label_name, chunk_idx) in latest:
    chunks_seen_per_label.setdefault(label_name, set()).add(chunk_idx)

EXPECTED_CHUNKS = 5329

print(f"{'Label':<28} {'Missing cells':>14} {'%':>7}   {'Never attempted':>16}")
for col in label_cols:
    n_missing = wide[col].isna().sum()
    n_seen = len(chunks_seen_per_label.get(col, set()))
    never_attempted = EXPECTED_CHUNKS - n_seen
    flag = "  ⚠️" if never_attempted > 20 else ""
    print(f"{col:<28} {n_missing:>14,} {n_missing/len(wide):>6.2%}   {never_attempted:>16,}{flag}")

Label                         Missing cells       %    Never attempted
Workplace Problem Guidance                1  0.00%                  0
Career Planning Advice                    1  0.00%                  0
Emotional Support                        11  0.02%                  1
Critical Pushback                       401  0.75%                 29  ⚠️
Personal Relating                        91  0.17%                  6
Discussion Direction                  1,571  2.95%                117  ⚠️
Practical Advice                         50  0.09%                  5
Support Connections                     131  0.25%                  7
informational_support                    21  0.04%                  2
emotional_support                        51  0.10%                  3
esteem_support                          151  0.28%                  9
tangible_support                         61  0.11%                  4
network_support                          61  0.11%                  6
unsupportiv